In [3]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# --- SETUP CELL (load + recreate derived columns) ---
inv = pd.read_csv("../Data/Raw/Inventory_History.csv", encoding="utf-8-sig")

# THIS is the line that was missing:
inv["Had_Stockout"] = (inv["Stockout_Days_In_Month"] > 0).astype(int)

inv["Snapshot_Month"] = pd.to_datetime(inv["Snapshot_Month"])
inv = inv.sort_values(["Location_ID", "Product_ID", "Snapshot_Month"])

# --- Lagged clues: LAST month's numbers only ---
lag_cols = ["Opening_Stock", "Days_of_Cover", "Avg_Daily_Sales_90D", "Sold_Qty"]
for c in lag_cols:
    inv["LAST_" + c] = inv.groupby(["Location_ID", "Product_ID"])[c].shift(1)

# --- Build X and y ---
features_lag = ["LAST_" + c for c in lag_cols] + ["Supplier_Lead_Time_Days", "Safety_Stock_Units"]
df = inv.dropna(subset=features_lag)
X = df[features_lag]
y = df["Had_Stockout"]

# --- Time-based exam: study on old months, exam on new months ---
train_mask = df["Snapshot_Month"] < "2026-01-01"
print("Study rows:", int(train_mask.sum()), "| Exam rows:", int((~train_mask).sum()))
print("Exam-month stockout rate:", round(y[~train_mask].mean(), 4))

rf_lag = RandomForestClassifier(n_estimators=100, max_depth=6,
                                random_state=42, class_weight="balanced")
rf_lag.fit(X[train_mask], y[train_mask])

prob = rf_lag.predict_proba(X[~train_mask])[:, 1]
print("FUTURE-MODEL ROC-AUC:", round(roc_auc_score(y[~train_mask], prob), 3))

print("\nWhat drives FUTURE stockouts:")
print(pd.Series(rf_lag.feature_importances_, index=features_lag)
        .sort_values(ascending=False).round(3))

Study rows: 4602 | Exam rows: 1593
Exam-month stockout rate: 0.0452
FUTURE-MODEL ROC-AUC: 0.95

What drives FUTURE stockouts:
LAST_Days_of_Cover          0.488
LAST_Opening_Stock          0.206
LAST_Sold_Qty               0.145
LAST_Avg_Daily_Sales_90D    0.100
Safety_Stock_Units          0.057
Supplier_Lead_Time_Days     0.005
dtype: float64


In [4]:
import joblib
from pathlib import Path

# Create the models folder if it doesn't exist
Path("../models").mkdir(exist_ok=True)

# Save the future-forecasting brain
joblib.dump(rf_lag, "../models/stockout_forecast_model.pkl")
print("Saved the Forecasting Brain to: models/stockout_forecast_model.pkl")

Saved the Forecasting Brain to: models/stockout_forecast_model.pkl
